# Car model analysis through TV shows

## Introduction

This project aims to analyze the appearance frequency of car models across various TV shows using web scraping techniques. The primary objective is to extract car data from the Internet Movie Cars Database (IMCDb.org) for a curated list of TV series, process this data, and identify popular car models and their most frequent appearances in specific shows.

**The key objectives covered in this notebook are:**

1.  **Environment Setup**: Install and configure Selenium WebDriver for web scraping in a Colab environment.
2.  **Demonstrative Scraping**: Perform example scraping runs for individual TV shows ("The Walking Dead", "Breaking Bad") to illustrate the core scraping logic.
3.  **Data Import & Preparation**: Load a comprehensive list of TV show titles from a Google Sheet and prepare a DataFrame for storing scraped car data.
4.  **Automated Data Collection**: Systematically iterate through the TV show list, navigate to each show's car inventory page on IMCDb.org (handling search, navigation, and potential missing links), and scrape all associated car models. This step includes robust error handling and driver re-initialization for stability.
5.  **Car Appearance Analysis**: Aggregate all collected car models and count their total appearances across all TV shows.
6.  **Top Car Identification**: Identify the top 10 most frequently appearing car models.
7.  **Series Appearance Breakdown**: For each of the top 10 cars, determine in which TV series they appeared most frequently and list all series they were featured in.
8.  **Data Export**: Save the processed results, including the full scraped dataset and the top 10 car analysis, into downloadable CSV files.

## Downloading selenium

In [ ]:
import sys

# Install selenium and webdriver_manager
!{sys.executable} -m pip install selenium webdriver_manager

# Install Google Chrome manually by downloading the .deb package
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
!apt-get update
!apt-get install -y google-chrome-stable

from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

# Set up Chrome options for running headless in Colab
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')       # Run in background without a visible UI
chrome_options.add_argument('--no-sandbox')     # Required for running as root in Colab
chrome_options.add_argument('--disable-dev-shm-usage') # Overcome limited resource problems
# Explicitly set the binary location for Google Chrome
chrome_options.binary_location = '/usr/bin/google-chrome'

# Initialize the WebDriver
# ChromeDriverManager automatically downloads and manages the correct driver for your Chrome version
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)

print("Selenium WebDriver initialized successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,212 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.co

## First example: "The walking dead"

### Searching for "The walking dead" and displaying the URL after the search
This code cell automates the process of navigating to the IMCDb.org website, handling potential cookie consent, and performing a search for a specified TV show.

- **Import Statements**: Imports necessary modules from `selenium` for web interaction, `time` for pauses.
- **Navigate to Homepage**: Opens the IMCDb homepage using `driver.get(homepage_url)`.
- **Handle Cookie Consent**: Attempts to find and click an 'I accept cookies' button within a 5-second timeout. This makes the script more robust to cookie banners.
- **Perform Search**:
    - Locates the search input field by its ID (`QsearchTitle`).
    - Clears any pre-existing text in the search field.
    - Enters the `search_query` (e.g., 'The walking dead').
    - Locates the search button by its class (`inputSubmit`).
    - Clicks the search button using JavaScript (`driver.execute_script("arguments[0].click();", search_button)`) for more reliable clicking, especially when elements might be obscured.
- **Pause and Verify**: A 2-second pause (`time.sleep(2)`) is used to allow the page to load after the search, then prints the current URL to confirm navigation.

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time # Import time for a brief pause

homepage_url = 'https://www.imcdb.org/'

# Navigate to the IMCDb homepage
driver.get(homepage_url)
print(f"Navigated to IMCDb homepage: {homepage_url}")

# --- Handle potential cookie consent banner ---
try:
    # Wait for the "I accept cookies" button to be clickable
    # The XPath targets a button with specific text, which is a common pattern for cookie consents.
    cookie_accept_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'I accept cookies')]"))
    )
    cookie_accept_button.click()
    print("Clicked 'I accept cookies' button.")
except (TimeoutException, NoSuchElementException):
    print("No cookie consent banner found or it was not clickable within the timeout.")
    # Continue if the banner is not found or already dismissed

search_query = 'The walking dead'
print(f"Searching for: {search_query}")

# Wait for the search input field to be visible and locate it by its ID 'QsearchTitle'
search_input = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, 'QsearchTitle'))
)

# Clear any existing text in the search input field
search_input.clear()

# Enter the search query into the input field
search_input.send_keys(search_query)

# Wait for the search button to be clickable and locate it by its class 'inputSubmit'
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CLASS_NAME, 'inputSubmit'))
)

# Click the search button using JavaScript to avoid ElementClickInterceptedException
driver.execute_script("arguments[0].click();", search_button)
print("Clicked search button using JavaScript.")

# Add a brief pause and print the current URL for debugging
time.sleep(2) # Give some time for the page to potentially navigate
print(f"Current URL immediately after click: {driver.current_url}")

Navigated to IMCDb homepage: https://www.imcdb.org/
No cookie consent banner found or it was not clickable within the timeout.
Searching for: The walking dead
Clicked search button using JavaScript.
Current URL immediately after click: https://www.imcdb.org/movies.php?title=The+walking+dead


### Extracting the URL of the TV show

This code cell is responsible for extracting the URL of a TV series from the search results obtained after performing a search on IMCDb.org.

- **HTML Parsing**: It retrieves the current page's HTML content using `driver.page_source` and parses it with `BeautifulSoup` to enable easy navigation and extraction of information.
- **Locate Search Results**: It identifies the main container for movie/TV series results by finding the `div` element with `id='Movies'`.
- **Iterate and Filter**: Within the `Movies` div, it looks for a `table` element and then iterates through its rows (`<tr>`).
- **Identify TV Series**: For each row, it checks if the entry is a 'TV Series' by examining the text content of the third `<td>` element. This helps distinguish TV series from movies or other types of entries.
- **Extract URL**: Once a 'TV Series' is identified, it extracts the `href` attribute from the `<a>` tag within the second `<td>` element, which corresponds to the link for that TV series. This `href` attribute (e.g., `movie_1520211-The-Walking-Dead.html`) is stored in `target_href_attribute`.
- **Error Handling**: It includes checks to see if the `Movies` div or the `car_table` were found, and if a TV series result was successfully identified, printing informative messages otherwise.

In [ ]:
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import re

# Get the page source after the search results have loaded
updated_html_content = driver.page_source
soup = BeautifulSoup(updated_html_content, 'html.parser')

# Find the div with id='Movies'
movies_div = soup.find('div', id='Movies')

if movies_div:
    # Find the table within the 'Movies' div
    car_table = movies_div.find('table')

    if car_table:
        # Get all <tr> (table row) elements within the found table
        tr_elements = car_table.find_all('tr')

        found_tv_series = False
        target_href_attribute = None

        # Iterate through search results starting from the second row (index 1) to skip header
        for tr in tr_elements[1:]:
            td_elements = tr.find_all('td')

            # Check if there are enough td elements to verify type (at least 3: image, title, type/year)
            if len(td_elements) > 2:
                # Check if the third <td> element (index 2) contains 'TV Series'
                if "TV Series" in td_elements[2].get_text(strip=True):
                    # Locate the <a> tag within the second <td> element (index 1)
                    a_tag = td_elements[1].find('a')
                    if a_tag and 'href' in a_tag.attrs:
                        target_href_attribute = a_tag['href']
                        found_tv_series = True
                        print(f"Found TV Series href for 'The walking dead': {target_href_attribute}")
                        break # Found the TV Series, exit this inner loop

        if not found_tv_series:
            print("No TV Series result found for 'The walking dead' in the search results.")

    else:
        print("Table not found within 'Movies' div.")
else:
    print("Div with ID 'Movies' not found on the page.")

Found TV Series href for 'The walking dead': movie_1520211-The-Walking-Dead.html


### Selecting the "Display as List" option
This code cell extracts a movie ID from a previously obtained `target_href_attribute` and uses it to construct a new URL that displays car information for the TV series in a list format. It then navigates the Selenium `driver` to this new URL.

- **Extract Movie ID**: It uses a regular expression (`re.search`) to find a numerical ID within the `target_href_attribute` string (e.g., `movie_1520211-The-Walking-Dead.html` will yield `1520211`).
- **Construct List View URL**: If an `movie_id` is successfully extracted, it constructs a specific URL for IMCDb.org that forces the display of car data as a list (`resultsStyle=asList`) and sorts it (`sortBy=0`).
- **Navigate WebDriver**: The `driver.get(list_view_url)` command directs the Selenium WebDriver to the newly constructed list view page.
- **Error Handling**: If the movie ID cannot be extracted, it prints an informative message indicating the failure.

In [ ]:
import re

# Extract the movie ID from the target_href_attribute
# The target_href_attribute should be like 'movie_1520211-The-Walking-Dead.html'
match = re.search(r'movie_(\d+)', target_href_attribute)
movie_id = None
if match:
    movie_id = match.group(1)

# Construct the correct URL for 'Display as list' with sortBy=0
if movie_id:
    list_view_url = f"https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id={movie_id}"
    driver.get(list_view_url)
    print(f"Navigated to 'The walking dead' TV series in list view: {driver.current_url}")
else:
    print("Could not extract movie ID from target_href_attribute. Cannot construct list view URL.")

Navigated to 'The walking dead' TV series in list view: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=1520211


### Using `BeautifulSoup` to scrape the car models
This code cell is designed to scrape car names from the currently loaded IMCDb.org page (which is expected to be a list view of a TV series' vehicles) and store them in a pandas DataFrame.

- **Retrieve HTML**: `driver.page_source` gets the full HTML content of the current web page, which is then parsed by `BeautifulSoup` for easy navigation.
- **Locate Vehicle Data Container**: It first searches for a `div` element with the ID `MovieVehicles`, which typically encloses all vehicle information on the page.
- **Refine Search to Box Contents**: Within `MovieVehicles`, it then looks for a `div` with the class `BoxContents`, which usually contains the main table of car listings.
- **Find Car Table**: It identifies the `table` element within `BoxContents` that holds the actual car data.
- **Extract Car Names**: All `<a>` (anchor) tags within this `car_table` are found. Each `<a>` tag's text is assumed to be a car name.
- **Clean Car Names**: For each extracted name, a regular expression (`re.sub`) is used to:
    - Remove any leading four-digit year (e.g., '2005 Ford Focus' becomes 'Ford Focus').
    - Remove any extra whitespace.
- **Populate DataFrame**: The cleaned car names are collected into a list (`car_names`), which is then converted into a pandas DataFrame called `example_df` with a single column 'Car Name'.
- **Error Handling**: The code includes `if` statements to check if the expected HTML elements (like `MovieVehicles` div, `BoxContents` div, and the `table`) are found, printing informative messages if they are not.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import re

# Retrieve the updated HTML content of the current page from the Selenium driver
updated_html_content = driver.page_source

# Create a new BeautifulSoup object with the updated HTML content
soup = BeautifulSoup(updated_html_content, 'html.parser')

# 1. Find the 'div' element with the ID 'MovieVehicles'
movie_vehicles_div = soup.find('div', id='MovieVehicles')

# Check if movie_vehicles_div was found
if movie_vehicles_div:
    # 2. Within movie_vehicles_div, find the 'div' with class 'BoxContents'
    box_contents_div = movie_vehicles_div.find('div', class_='BoxContents')

    # Check if box_contents_div was found
    if box_contents_div:
        # 3. Within box_contents_div, find the 'table' element
        car_table = box_contents_div.find('table')

        # Check if car_table was found
        if car_table:
            # 4. From the car_table, find all 'a' (anchor) tags
            car_name_tags = car_table.find_all('a')

            # 5. Initialize an empty list to store car names
            car_names = []

            # 6. Iterate through the found 'a' tags and extract the text
            for tag in car_name_tags:
                # Clean up the text by removing leading/trailing whitespace and multiple spaces
                name = re.sub(r'\s+', ' ', tag.text).strip()
                # Remove the leading four-digit year and any subsequent whitespace
                name = re.sub(r'^\d{4}\s*', '', name).strip()
                if name: # Only add if the name is not empty after cleaning
                    car_names.append(name)

            # 7. Create a pandas DataFrame from the list of car names
            example_df = pd.DataFrame(car_names, columns=['Car Name'])

            # 8. Display the DataFrame
            print("Extracted and cleaned car names (in example_df):")
            print(example_df)
        else:
            print("Table not found within 'BoxContents' div.")
    else:
        print("Div with class 'BoxContents' not found within 'MovieVehicles' div.")
else:
    print("Div with ID 'MovieVehicles' not found.")

Extracted and cleaned car names (in example_df):
                                              Car Name
0                                  Acura Integra [DB1]
1                                   Allis-Chalmers 160
2                                     AM General HMMWV
3                               AM General HMMWV M1037
4                               AM General HMMWV M1043
..                                                 ...
497                                     WhiteGMC WG 64
498  Winnebago Chieftain on Dodge RM400 chassis [D-...
499                              Winnebago Super Chief
500                                      Yamaha SR 500
501                                Yamaha XT 225 Serow

[502 rows x 1 columns]


## Second example: "Breaking Bad"

Now we perform the same actions for the Tv show "Breaking Bad"

In [ ]:
driver.get(homepage_url)
print(f"Navigated to IMCDb homepage: {homepage_url}")

Navigated to IMCDb homepage: https://www.imcdb.org/


In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time # Import time for a brief pause

search_query = 'Breaking Bad'
print(f"Searching for: {search_query}")

# --- Handle potential cookie consent banner (copied from previous code) ---
try:
    cookie_accept_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'I accept cookies')]" ))
    )
    cookie_accept_button.click()
    print("Clicked 'I accept cookies' button.")
except (TimeoutException, NoSuchElementException):
    print("No cookie consent banner found or it was not clickable within the timeout.")

# Wait for the search input field to be visible and locate it by its ID 'QsearchTitle'
search_input = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, 'QsearchTitle'))
)

# Clear any existing text in the search input field
search_input.clear()

# Enter the search query into the input field
search_input.send_keys(search_query)

# Wait for the search button to be clickable and locate it by its class 'inputSubmit'
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CLASS_NAME, 'inputSubmit'))
)

# Click the search button using JavaScript to avoid ElementClickInterceptedException
driver.execute_script("arguments[0].click();", search_button)
print("Clicked search button using JavaScript.")

# Add a brief pause and print the current URL for debugging
time.sleep(2) # Give some time for the page to potentially navigate
print(f"Current URL immediately after click: {driver.current_url}")

Searching for: Breaking Bad
No cookie consent banner found or it was not clickable within the timeout.
Clicked search button using JavaScript.
Current URL immediately after click: https://www.imcdb.org/movies.php?title=Breaking+Bad


In [ ]:
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import re

# Get the page source after the search results have loaded
updated_html_content = driver.page_source
soup = BeautifulSoup(updated_html_content, 'html.parser')

# Find the div with id='Movies'
movies_div = soup.find('div', id='Movies')

if movies_div:
    # Find the table within the 'Movies' div
    car_table = movies_div.find('table')

    if car_table:
        # Get all <tr> (table row) elements within the found table
        tr_elements = car_table.find_all('tr')

        found_tv_series_bb = False
        target_href_attribute_bb = None

        # Iterate through search results starting from the second row (index 1) to skip header
        for tr in tr_elements[1:]:
            td_elements = tr.find_all('td')

            # Check if there are enough td elements to verify type (at least 3: image, title, type/year)
            if len(td_elements) > 2:
                # Check if the third <td> element (index 2) contains 'TV Series'
                # And the title (second <td> element) contains 'Breaking Bad'
                if "TV Series" in td_elements[2].get_text(strip=True) and "Breaking Bad" in td_elements[1].get_text(strip=True):
                    # Locate the <a> tag within the second <td> element (index 1)
                    a_tag = td_elements[1].find('a')
                    if a_tag and 'href' in a_tag.attrs:
                        target_href_attribute_bb = a_tag['href']
                        found_tv_series_bb = True
                        print(f"Found TV Series href for 'Breaking Bad': {target_href_attribute_bb}")
                        break # Found the TV Series, exit this inner loop

        if not found_tv_series_bb:
            print("No TV Series result found for 'Breaking Bad' in the search results.")

    else:
        print("Table not found within 'Movies' div.")
else:
    print("Div with ID 'Movies' not found on the page.")

Found TV Series href for 'Breaking Bad': movie_903747-Breaking-Bad.html


In [ ]:
import re

# Extract the movie ID from the target_href_attribute_bb
# The target_href_attribute_bb should be like 'movie_903747-Breaking-Bad.html'
match = re.search(r'movie_(\d+)', target_href_attribute_bb)
movie_id_bb = None
if match:
    movie_id_bb = match.group(1)

# Construct the correct URL for 'Display as list' with sortBy=0
if movie_id_bb:
    list_view_url_bb = f"https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id={movie_id_bb}"
    driver.get(list_view_url_bb)
    print(f"Navigated to 'Breaking Bad' TV series in list view: {driver.current_url}")
else:
    print("Could not extract movie ID from target_href_attribute_bb. Cannot construct list view URL.")

Navigated to 'Breaking Bad' TV series in list view: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=903747


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import re

# Retrieve the updated HTML content of the current page from the Selenium driver
updated_html_content = driver.page_source

# Create a new BeautifulSoup object with the updated HTML content
soup = BeautifulSoup(updated_html_content, 'html.parser')

# 1. Find the 'div' element with the ID 'MovieVehicles'
movie_vehicles_div = soup.find('div', id='MovieVehicles')

# Check if movie_vehicles_div was found
if movie_vehicles_div:
    # 2. Within movie_vehicles_div, find the 'div' with class 'BoxContents'
    box_contents_div = movie_vehicles_div.find('div', class_='BoxContents')

    # Check if box_contents_div was found
    if box_contents_div:
        # 3. Within box_contents_div, find the 'table' element
        car_table = box_contents_div.find('table')

        # Check if car_table was found
        if car_table:
            # 4. From the car_table, find all 'a' (anchor) tags
            car_name_tags_bb = car_table.find_all('a')

            # 5. Initialize an empty list to store car names
            car_names_bb = []

            # 6. Iterate through the found 'a' tags and extract the text
            for tag in car_name_tags_bb:
                # Clean up the text by removing leading/trailing whitespace and multiple spaces
                name = re.sub(r'\s+', ' ', tag.text).strip()
                # Remove the leading four-digit year and any subsequent whitespace
                name = re.sub(r'^\d{4}\s*', '', name).strip()
                if name: # Only add if the name is not empty after cleaning
                    car_names_bb.append(name)

            # 7. Create a pandas DataFrame from the list of car names
            breaking_bad_cars_df = pd.DataFrame(car_names_bb, columns=['Car Name'])

            # 8. Display the DataFrame
            print("Extracted and cleaned car names for 'Breaking Bad' (in breaking_bad_cars_df):")
            print(breaking_bad_cars_df)
        else:
            print("Table not found within 'BoxContents' div for 'Breaking Bad'.")
    else:
        print("Div with class 'BoxContents' not found within 'MovieVehicles' div for 'Breaking Bad'.")
else:
    print("Div with ID 'MovieVehicles' not found for 'Breaking Bad'.")

Extracted and cleaned car names for 'Breaking Bad' (in breaking_bad_cars_df):
                        Car Name
0            AMC Eagle 4WD Wagon
1            Audi A4 B6 [Typ 8E]
2    Audi A4 quattro B7 [Typ 8E]
3                   Autocar S64U
4         Bentley Continental GT
..                           ...
404              Volvo 740 [745]
405              Volvo S40 Gen.2
406          Volvo V70 GLT Gen.1
407    Volvo XC60 R-Design Gen.1
408              Yamaha TT-R 125

[409 rows x 1 columns]


As we can see the cars from the show are present in the `breaking_bad_cars_df`, so now we can move on to the **main data collection task**.

## Main Data Collection

### Data Import and Preparation

This code snippet is responsible for importing TV show titles from a precollected Google Sheet and preparing a pandas DataFrame for analysis.

-   **Import pandas**: The `pandas` library is imported for data manipulation.
-   **Google Sheet URL**: Defines the URL of the Google Sheet containing the TV show data.
-   **Convert to CSV Export URL**: The `sheet_url` is transformed into a direct CSV export URL. This is done by replacing `/edit?gid=` with `/export?format=csv&gid=` and removing any hash anchors (`#`).
-   **Read CSV into DataFrame**: The data from the `csv_url` is read directly into a pandas DataFrame named `tv_show_cars_df`.
-   **Select 'Title' Column**: Only the 'Title' column is selected from the DataFrame, ensuring that only the TV show names are retained for further processing.
-   **Display DataFrame Head**: The `head()` method is used to print the first few rows of the `tv_show_cars_df`, allowing for a quick verification of the loaded and prepared data.

In [ ]:
import pandas as pd

# Google Sheet URL
sheet_url = "https://docs.google.com/spreadsheets/d/18o2nPmTGxYQDwRmroX-AEUdGRFDlEX5AtFuFFVTuLg0/edit?gid=17353310#gid=17353310"

# Convert to direct CSV export URL for the specified gid
csv_url = sheet_url.replace('/edit?gid=', '/export?format=csv&gid=').split('#')[0]

# Read the CSV data into a pandas DataFrame
tv_show_cars_df = pd.read_csv(csv_url)

# Select only the 'Title' column (assuming the column name is 'Title')
tv_show_cars_df = tv_show_cars_df[['Title']]

# Display the first few rows of the DataFrame to verify
print("DataFrame 'tv_show_cars_df' with only titles:")
print(tv_show_cars_df.head())

DataFrame 'tv_show_cars_df' with only titles:
             Title
0     Breaking Bad
1  The Night Agent
2         The Pitt
3  Stranger Things
4        Shrinking


### Adding 'Cars' Column

This code snippet adds a new column named 'Cars' to the `tv_show_cars_df` DataFrame. This column is initialized with an empty list `[]` for each row, which will later be populated with the car names scraped for each corresponding TV show.


In [ ]:
tv_show_cars_df['Cars'] = [[] for _ in range(len(tv_show_cars_df))]
print(tv_show_cars_df.head())

             Title Cars
0     Breaking Bad   []
1  The Night Agent   []
2         The Pitt   []
3  Stranger Things   []
4        Shrinking   []


### Iterate, Filter, Scrape, and Populate DataFrame

This code block iterates through each TV show title in the `tv_show_cars_df` DataFrame, performing the following steps for each show:

-   **Initialize `list_view_url_temp`**: A new column `list_view_url_temp` is added to `tv_show_cars_df` to store the determined list view URL for each show. It's initialized to `None`.
-   **Retry Mechanism**: Each show's processing is wrapped in a `while` loop with `MAX_RETRIES` to handle transient network issues or page loading errors.
-   **Navigate to Homepage**: The Selenium `driver` navigates to the IMCDb homepage (`homepage_url`).
-   **Handle Cookie Consent**: It attempts to click an 'I accept cookies' button if a cookie consent banner appears, making the script more robust.
-   **Perform Search**: It locates the search input field (`QsearchTitle`), clears it, enters the TV show `title`, and clicks the search button (`inputSubmit`).
-   **Parse Search Results**: After a brief pause to allow the page to load, `BeautifulSoup` parses the current page's HTML to find the 'Movies' section and its `table` of search results.
-   **Identify TV Series Link**: It iterates through the search results to find an entry that matches the current TV show `title` (case-insensitive and partial match) and is identified as a 'TV Series'.
-   **Extract Movie ID and Construct URL**: Once a matching TV Series is found, its `href` attribute is extracted, and a regular expression (`re.search`) is used to get the numeric `movie_id`. This `movie_id` is then used to construct the `list_view_url` for the 'Display as List' view of the cars for that series.
-   **Navigate to List View**: The `driver` navigates to the constructed `list_view_url`.
-   **Update DataFrame**: If navigation is successful, the `list_view_url` is stored in the `list_view_url_temp` column for the corresponding TV show. If all retries fail, 'Failed' is stored instead.
-   **Error Handling**: Comprehensive `try-except` blocks are used to catch `TimeoutException`, `NoSuchElementException`, and general `Exception` during the web scraping process, logging errors and retrying up to `MAX_RETRIES` times.

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import time
import re

# Add a temporary column to store the list view URLs for verification
tv_show_cars_df['list_view_url_temp'] = None

# Define max retries for robust error handling
MAX_RETRIES = 3

print("Starting scraping process for each TV show...")

# Iterate through each TV show title in the DataFrame
for index, row in tv_show_cars_df.iterrows():
    title = row['Title']
    current_retry = 0
    successful_navigation = False
    list_view_url = None # Initialize to None for each show

    print(f"\nProcessing '{title}' (Row {index})...")

    while current_retry < MAX_RETRIES:
        try:
            # 1. Navigate to the IMCDb homepage
            driver.get(homepage_url)
            # print(f"Navigated to IMCDb homepage for '{title}': {homepage_url}")

            # 2. Handle potential cookie consent banner
            try:
                cookie_accept_button = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'I accept cookies')]"))
                )
                cookie_accept_button.click()
                print("  Clicked 'I accept cookies' button.")
            except (TimeoutException, NoSuchElementException):
                # print("  No cookie consent banner found or it was not clickable within the timeout.")
                pass # Continue if banner not found or already dismissed

            # 3. Wait for the search input field and enter query
            search_input = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.ID, 'QsearchTitle'))
            )
            search_input.clear()
            search_input.send_keys(title)
            # print(f"  Entered search query: '{title}'")

            # 4. Wait for and click the search button
            search_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CLASS_NAME, 'inputSubmit'))
            )
            driver.execute_script("arguments[0].click();", search_button)
            # print("  Clicked search button.")

            # 5. Add a brief pause for page navigation
            time.sleep(2)

            # 6. Get the page source and parse with BeautifulSoup
            updated_html_content = driver.page_source
            soup = BeautifulSoup(updated_html_content, 'html.parser')

            # 7. Find the div with id='Movies' and the table within it
            movies_div = soup.find('div', id='Movies')
            car_table = movies_div.find('table') if movies_div else None

            if car_table:
                tr_elements = car_table.find_all('tr')
                target_href_attribute = None

                # Iterate through search results to find the correct TV Series
                for tr in tr_elements[1:]:
                    td_elements = tr.find_all('td')
                    if len(td_elements) > 2:
                        # Check if it's a TV Series and title matches (case-insensitive and partial match)
                        if "TV Series" in td_elements[2].get_text(strip=True) and \
                           title.lower() in td_elements[1].get_text(strip=True).lower():
                            a_tag = td_elements[1].find('a')
                            if a_tag and 'href' in a_tag.attrs:
                                target_href_attribute = a_tag['href']
                                break

                if target_href_attribute:
                    # 8. Extract movie_id and construct list_view_url
                    match = re.search(r'movie_(\d+)', target_href_attribute)
                    if match:
                        movie_id = match.group(1)
                        list_view_url = f"https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id={movie_id}"

                        # 9. Navigate the driver to the list view URL
                        driver.get(list_view_url)
                        print(f"  Successfully navigated to list view for '{title}': {driver.current_url}")
                        successful_navigation = True
                        break # Exit retry loop on success
                    else:
                        print(f"  Could not extract movie ID for '{title}'.")
                else:
                    print(f"  No matching TV Series link found for '{title}'.")

            else:
                print(f"  Car table not found within 'Movies' div for '{title}'.")

        except (TimeoutException, NoSuchElementException) as e:
            print(f"  Attempt {current_retry + 1}/{MAX_RETRIES} failed for '{title}' (Selenium error: {e}). Retrying...")
        except Exception as e:
            print(f"  Attempt {current_retry + 1}/{MAX_RETRIES} failed for '{title}' (General error: {e}). Retrying...")

        current_retry += 1
        time.sleep(1) # Small delay before retrying

    if successful_navigation:
        tv_show_cars_df.loc[index, 'list_view_url_temp'] = list_view_url
    else:
        print(f"  Failed to navigate to list view for '{title}' after {MAX_RETRIES} attempts.")
        tv_show_cars_df.loc[index, 'list_view_url_temp'] = 'Failed'

print("\nNavigation process complete.")
print("Updated DataFrame with navigation status (first 5 rows):")
print(tv_show_cars_df.head())

Starting scraping process for each TV show...

Processing 'Breaking Bad' (Row 0)...
  Successfully navigated to list view for 'Breaking Bad': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=903747

Processing 'The Night Agent' (Row 1)...
  Successfully navigated to list view for 'The Night Agent': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=13918776

Processing 'The Pitt' (Row 2)...
  Successfully navigated to list view for 'The Pitt': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=31938062

Processing 'Stranger Things' (Row 3)...
  Successfully navigated to list view for 'Stranger Things': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=4574334

Processing 'Shrinking' (Row 4)...
  Successfully navigated to list view for 'Shrinking': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=15677150

Processing 'The Night Manager' (Row 5)...
  Successfully navigated to list view for 'The Night Manager': https://

As a result a new collumn in the `tv_show_cars_df` dataframe with the corresponding URLs for each show is created.

In [ ]:
tv_show_cars_df

,Title,Cars,list_view_url_temp
0,Breaking Bad,[],https://www.imcdb.org/movie.php?resultsStyle=a...
1,The Night Agent,[],https://www.imcdb.org/movie.php?resultsStyle=a...
2,The Pitt,[],https://www.imcdb.org/movie.php?resultsStyle=a...
3,Stranger Things,[],https://www.imcdb.org/movie.php?resultsStyle=a...
4,Shrinking,[],https://www.imcdb.org/movie.php?resultsStyle=a...
...,...,...,...
60,Supernatural,[],https://www.imcdb.org/movie.php?resultsStyle=a...
61,Severance,[],https://www.imcdb.org/movie.php?resultsStyle=a...
62,Homeland,[],https://www.imcdb.org/movie.php?resultsStyle=a...
63,Dark,[],https://www.imcdb.org/movie.php?resultsStyle=a...


### Finding the missing links
In this cell we search for the TV shows, for which the URL was not found. For these rows, we will add the URLs manually.

In [ ]:
failed_links_df = tv_show_cars_df[tv_show_cars_df['list_view_url_temp'] == 'Failed']
print("TV show titles where list_view_url_temp is 'Failed':")
print(failed_links_df[['Title', 'list_view_url_temp']])

TV show titles where list_view_url_temp is 'Failed':
              Title list_view_url_temp
9           Landman             Failed
10       His & Hers             Failed
12         Pluribus             Failed
20       Smallville             Failed
23  The Beast in Me             Failed
41       The Bureau             Failed
42     Mr Inbetween             Failed
44   The Family Man             Failed
45          Fleabag             Failed


### Documentation: Finding and Updating Missing Links

-   **`updated_links` dictionary**: The links, that are collected manually, are gathered in `updated_links` dataframe.
-   **Updating DataFrame**: The code then iterates through this `updated_links` dictionary.
    -   For each `title` and `url` pair, it uses `tv_show_cars_df.loc[tv_show_cars_df['Title'] == title, 'list_view_url_temp'] = url` to locate the corresponding row in the `tv_show_cars_df` DataFrame by `Title` and update its `list_view_url_temp` column with the correct URL.

In [ ]:
updated_links = {
    'Landman': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=14186672',
    'His & Hers': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=33035373',
    'Pluribus': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=22202452',
    'Smallville': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=279600',
    'The Beast in Me': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=31974367',
    'The Bureau': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=4063800',
    'Mr Inbetween': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=7472896',
    'The Family Man': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=218967',
    'Fleabag': 'https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=5687612'
}

# Update the 'list_view_url_temp' column in tv_show_cars_df
for title, url in updated_links.items():
    tv_show_cars_df.loc[tv_show_cars_df['Title'] == title, 'list_view_url_temp'] = url
    print(f"Updated link for '{title}' to: {url}")

print("\nDataFrame 'tv_show_cars_df' after updating specific links:")
print(tv_show_cars_df.head(15)) # Display a few more rows to show updated entries

Updated link for 'Landman' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=14186672
Updated link for 'His & Hers' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=33035373
Updated link for 'Pluribus' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=22202452
Updated link for 'Smallville' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=279600
Updated link for 'The Beast in Me' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=31974367
Updated link for 'The Bureau' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=4063800
Updated link for 'Mr Inbetween' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=7472896
Updated link for 'The Family Man' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=218967
Updated link for 'Fleabag' to: https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=5687612

DataFrame 'tv_show_cars_df' after updati

### Refined Scraping Logic (using `list_view_url_temp`)

This code block iterates through each row of the `tv_show_cars_df` DataFrame, directly utilizing the URLs stored in the `list_view_url_temp` column to navigate the Selenium WebDriver and scrape car names for each TV show. This refined approach bypasses the initial search steps, making the scraping process more efficient.

-   **`initialize_driver()` function**: A utility function is defined to set up and re-initialize the Selenium WebDriver (Chrome in headless mode). It includes logic to quit any existing driver instances before creating a new one, ensuring a clean state and addressing potential `WebDriverException` or `ConnectionError`.
-   **Initial Driver Check**: Before the main loop, there's a check to ensure the `driver` instance is active and responsive. If not, or if it's the first run, the `initialize_driver()` function is called.
-   **Iterate Through DataFrame**: The code loops through each `title` and its corresponding `list_view_url` in `tv_show_cars_df`.
-   **Handle 'Failed' URLs**: If `list_view_url` is marked as 'Failed' or is `None`, the row is skipped, and the 'Cars' column for that entry is set to an empty list.
-   **Retry Mechanism**: Each show's processing is encapsulated in a `while` loop with `MAX_RETRIES_PER_SHOW` to handle transient network issues, page loading problems, or WebDriver instability.
-   **Direct Navigation**: Instead of performing a search, the `driver` directly navigates to the `list_view_url` (e.g., `driver.get(list_view_url)`).
-   **Handle Cookie Consent**: A check for the cookie consent banner is included after navigation, as it might reappear.
-   **Scrape Car Names**: `BeautifulSoup` parses the page source, locates the relevant HTML elements (div with `id='MovieVehicles'`, div with `class='BoxContents'`, and the car `table`), and extracts all car names within `<a>` tags.
-   **`clean_car_name()` function**: This function (re-defined or imported) is used to preprocess each scraped car name, typically removing leading years and text within square brackets for standardization.
-   **Populate 'Cars' Column**: The cleaned car names for each show are appended to a `cars_for_show` list, which is then assigned to the 'Cars' column of the respective row in `tv_show_cars_df`.
-   **Robust Error Handling and Re-initialization**: `try-except` blocks are used to catch various exceptions (`NoSuchElementException`, `TimeoutException`, `WebDriverException`, `ConnectionError`, `MaxRetryError`, and general `Exception`). If a `WebDriverException`, `ConnectionError`, or `MaxRetryError` occurs, the `driver` is quit and re-initialized to attempt to recover from connection or browser issues.
-   **Close WebDriver**: Finally, `driver.quit()` is called to properly close the Selenium WebDriver instance, releasing system resources.

In [ ]:
import re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException
from requests.exceptions import ConnectionError # Import ConnectionError for broader network issues
from urllib3.exceptions import MaxRetryError # Import MaxRetryError explicitly
import time

# Define the driver initialization function
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.binary_location = '/usr/bin/google-chrome'
    global driver # Declare driver as global to modify the global variable

    # Try to clean up any existing driver process before starting a new one
    try:
        if 'driver' in globals() and driver:
            driver.quit()
            time.sleep(1) # Give a moment for the port to release
    except Exception as e:
        print(f"  Error during pre-initialization cleanup of driver: {e}")

    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    print("Selenium WebDriver re-initialized successfully.")
    return driver

# Ensure driver is initialized at the start of the whole process or re-initialize if needed.
# This block attempts to ensure a working driver is available before the main loop.
try:
    # Try a simple operation that would fail if the driver is disconnected
    # Navigating to a known URL is more robust than just checking .current_url
    if 'driver' not in globals() or not driver:
        # If driver not defined or is None, initialize it
        driver = initialize_driver()
    else:
        # If driver exists, check if it's still functional by navigating
        driver.get("about:blank") # Navigate to a blank page to test connectivity
        print("Existing Selenium WebDriver is responsive.")
except (NameError, WebDriverException, ConnectionError, MaxRetryError) as e:
    print(f"Initial driver instance is not active or failed to respond ({type(e).__name__}: {e}). Re-initializing once.")
    driver = initialize_driver()

# Function to clean car names (re-using logic from previous steps)
def clean_car_name(name):
    # Fix: remove escaped backslashes in regex patterns
    name = re.sub(r'^\d{4}\s*', '', name).strip()
    name = re.sub(r'\s*\[[^\]]*\]', '', name).strip()
    return name


print("Starting car scraping process for each TV show using direct URLs...")

# Iterate through each TV show title in the tv_show_cars_df
for index, row in tv_show_cars_df.iterrows():
    title = row['Title']
    list_view_url = row['list_view_url_temp']
    cars_for_show = [] # Initialize list to store cars for the current show

    print(f"\nProcessing: {title}")

    # Skip if the URL is marked as 'Failed'
    if list_view_url == 'Failed' or list_view_url is None:
        print(f"  Skipping '{title}' due to 'Failed' or empty URL.")
        tv_show_cars_df.at[index, 'Cars'] = [] # Ensure 'Cars' column is an empty list for failed entries
        continue

    # Use a retry mechanism for each show in case of transient WebDriver issues
    MAX_RETRIES_PER_SHOW = 3
    current_attempt = 0
    last_exception = None
    successful_processing = False

    while current_attempt < MAX_RETRIES_PER_SHOW and not successful_processing:
        try:
            # 1. Navigate directly to the list view URL
            driver.get(list_view_url)
            print(f"  Navigated to list view for '{title}': {driver.current_url}")

            # 2. Handle potential cookie consent banner (using existing logic from previous tasks)
            # This part should be after every driver.get() as the cookie banner might reappear.
            try:
                cookie_accept_button = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'I accept cookies')]" ))
                )
                cookie_accept_button.click()
                print("    Clicked 'I accept cookies' button.")
            except (TimeoutException, NoSuchElementException):
                pass # Continue if banner not found or already dismissed

            # 3. Retrieve updated HTML and parse it for car names
            updated_html_content_list_view = driver.page_source
            soup_list_view = BeautifulSoup(updated_html_content_list_view, 'html.parser')

            movie_vehicles_div_list_view = soup_list_view.find('div', id='MovieVehicles')

            if movie_vehicles_div_list_view:
                box_contents_div_list_view = movie_vehicles_div_list_view.find('div', class_='BoxContents')

                if box_contents_div_list_view:
                    car_data_table_list_view = box_contents_div_list_view.find('table')

                    if car_data_table_list_view:
                        car_name_tags = car_data_table_list_view.find_all('a')
                        for tag in car_name_tags:
                            name = clean_car_name(tag.text) # Use the cleaning function
                            if name: # Only add if the name is not empty after cleaning
                                cars_for_show.append(name)
                        print(f"  Extracted {len(cars_for_show)} car names.")
                    else:
                        print(f"  Table not found within 'BoxContents' div for '{title}'.")
                else:
                    print(f"  Div with class 'BoxContents' not found within 'MovieVehicles' div for '{title}'.")
            else:
                print(f"  Div with ID 'MovieVehicles' not found on the list view page for '{title}'.")

            successful_processing = True # Mark as successful if we reach here without exceptions

        except (NoSuchElementException, TimeoutException) as e:
            print(f"  Attempt {current_attempt + 1}/{MAX_RETRIES_PER_SHOW} failed for '{title}' (Selenium element error: {type(e).__name__}: {e}). Retrying...")
            last_exception = e
        except (WebDriverException, ConnectionError, MaxRetryError) as e:
            print(f"  Attempt {current_attempt + 1}/{MAX_RETRIES_PER_SHOW} failed for '{title}' (WebDriver connection/network error: {type(e).__name__}: {e}). Re-initializing driver...")
            last_exception = e
            try:
                driver.quit() # Clean up defunct driver
            except Exception as e_quit:
                print(f"  Error quitting driver: {e_quit}")
            driver = initialize_driver() # Re-initialize the driver
        except Exception as e:
            print(f"  Attempt {current_attempt + 1}/{MAX_RETRIES_PER_SHOW} failed for '{title}' (General error: {type(e).__name__}: {e}). Retrying...")
            last_exception = e

        current_attempt += 1
        if not successful_processing and current_attempt < MAX_RETRIES_PER_SHOW:
            time.sleep(2) # Small delay before retrying

    if not successful_processing:
        print(f"Failed to process '{title}' after {MAX_RETRIES_PER_SHOW} attempts. Last error: {type(last_exception).__name__}: {last_exception}")

    # Assign the car_names list to the 'Cars' column for the current row
    tv_show_cars_df.at[index, 'Cars'] = cars_for_show

# Display the updated tv_show_cars_df
print("\nUpdated tv_show_cars_df:")
display(tv_show_cars_df)

# Close the WebDriver after all scraping is done
# Ensure driver is quit even if an error occurred in the loop but not caught by outer try-except for the entire script.
try:
    if 'driver' in globals() and driver:
        driver.quit()
        print("WebDriver closed.")
except Exception as e:
    print(f"Error closing WebDriver: {e}")

Initial driver instance is not active or failed to respond (MaxRetryError: HTTPConnectionPool(host='localhost', port=44415): Max retries exceeded with url: /session/a24aa8174863909d69c9be5009a166a9/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=44415): Failed to establish a new connection: [Errno 111] Connection refused"))). Re-initializing once.
Selenium WebDriver re-initialized successfully.
Starting car scraping process for each TV show using direct URLs...

Processing: Breaking Bad
  Navigated to list view for 'Breaking Bad': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=903747
  Extracted 409 car names.

Processing: The Night Agent
  Navigated to list view for 'The Night Agent': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=13918776
  Extracted 68 car names.

Processing: The Pitt
  Navigated to list view for 'The Pitt': https://www.imcdb.org/movie.php?resultsStyle=asList&sortBy=0&id=31938062
  Extracted 30 car names.

P

,Title,Cars,list_view_url_temp
0,Breaking Bad,"[AMC Eagle 4WD Wagon, Audi A4 B6, Audi A4 quat...",https://www.imcdb.org/movie.php?resultsStyle=a...
1,The Night Agent,"[AM General HMMWV M1025, BMW 2, BMW X1 xDrive,...",https://www.imcdb.org/movie.php?resultsStyle=a...
2,The Pitt,"[Audi Q3, Buick LeSabre, Chevrolet Cruze, Chev...",https://www.imcdb.org/movie.php?resultsStyle=a...
3,Stranger Things,"[AM General DJ-5 Dispatcher, AM General HMMWV ...",https://www.imcdb.org/movie.php?resultsStyle=a...
4,Shrinking,"[Audi A4 B8, Audi A5 2.0 TFSI B8, Audi A6 C8, ...",https://www.imcdb.org/movie.php?resultsStyle=a...
...,...,...,...
60,Supernatural,"[Acura Integra, Acura RDX, Acura RSX, AM Gener...",https://www.imcdb.org/movie.php?resultsStyle=a...
61,Severance,"[BMW 3, Buick Riviera, Buick Skylark, Chevrole...",https://www.imcdb.org/movie.php?resultsStyle=a...
62,Homeland,"[Acura TL Type-S, AM General HMMWV M1038, AM G...",https://www.imcdb.org/movie.php?resultsStyle=a...
63,Dark,"[Audi A3 Sportback g-tron, Audi A6 3.0 TDI qua...",https://www.imcdb.org/movie.php?resultsStyle=a...


WebDriver closed.


### Saving the results
The result is saved and downloaded into `tv_shows_with_cars.csv`

In [ ]:
from google.colab import files

# Convert the DataFrame to a CSV string
csv_data = tv_show_cars_df.to_csv(index=False)

# Save the CSV data to a file
with open('tv_shows_with_cars.csv', 'w') as f:
    f.write(csv_data)

# Download the file
files.download('tv_shows_with_cars.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Counting Car Appearances

This code block processes the scraped car data to determine the overall appearance frequency of each car model across all TV shows.

-   **Collect All Car Names**: It iterates through the 'Cars' column of `tv_show_cars_df`, which contains a list of cars for each TV show. It uses `all_cars.extend(car_list)` to flatten all these individual lists into a single `all_cars` list, containing every car mentioned in every show.
-   **Count Appearances**: `collections.Counter(all_cars)` is used to efficiently count the occurrences of each unique car model within the `all_cars` list.
-   **Create DataFrame**: A new pandas DataFrame, `car_appearances_df`, is created from the `car_counts`. This DataFrame has two columns: 'Car Model' (for the unique car names) and 'Appearances' (for their respective counts).
-   **Sort DataFrame**: The `car_appearances_df` is then sorted in descending order based on the 'Appearances' column, with `reset_index(drop=True)` to clean up the index after sorting.


In [ ]:
from collections import Counter
import pandas as pd

# 1. Collect all car names from the 'Cars' column into a single list
all_cars = []
for car_list in tv_show_cars_df['Cars']:
    all_cars.extend(car_list)

# 2. Count the appearances of each car model
car_counts = Counter(all_cars)

# 3. Create a DataFrame from the car_counts
car_appearances_df = pd.DataFrame(car_counts.items(), columns=['Car Model', 'Appearances'])

# 4. Sort the DataFrame by 'Appearances' in descending order
car_appearances_df = car_appearances_df.sort_values(by='Appearances', ascending=False).reset_index(drop=True)

# 5. Display the new DataFrame
print("DataFrame 'car_appearances_df' showing car model appearances:")
print(car_appearances_df)

DataFrame 'car_appearances_df' showing car model appearances:
                               Car Model  Appearances
0                                unknown          173
1                           Toyota Camry          127
2                            Honda Civic          126
3                         Toyota Corolla          118
4                         Ford Econoline          113
...                                  ...          ...
5697          Toyota Truck Itasca Spirit            1
5698  Toyota Tacoma PreRunner Double Cab            1
5699           Toyota Tacoma Regular Cab            1
5700         Toyota Corolla Hatchback SE            1
5701          Toyota 4Runner Limited 4WD            1

[5702 rows x 2 columns]


Saving the `car_appearances_df` dataframe into a `car_appearances.csv`

In [ ]:
from google.colab import files

# Convert the DataFrame to a CSV string
csv_data_appearances = car_appearances_df.to_csv(index=False)

# Save the CSV data to a file
with open('car_appearances.csv', 'w') as f:
    f.write(csv_data_appearances)

# Download the file
files.download('car_appearances.csv')

print("DataFrame 'car_appearances.csv' downloaded successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DataFrame 'car_appearances.csv' downloaded successfully.


## Identify Top 10 Cars

Now to analyze the most frequent cars, we need to select the top 10 cars. At first we must remove the first row, as the car model for that row is **uknown**.


In [ ]:
car_appearances_df = car_appearances_df.iloc[1:].reset_index(drop=True)


Then we collect the top 10 cars into `top_10_cars_df` dataframe.

In [ ]:
top_10_cars_df = car_appearances_df.head(10)

print("Top 10 car models based on appearances:")
print(top_10_cars_df)

Top 10 car models based on appearances:
                                Car Model  Appearances
0                            Toyota Camry          127
1                             Honda Civic          126
2                          Toyota Corolla          118
3                          Ford Econoline          113
4  Ford Crown Victoria Police Interceptor          102
5                            Ford Mustang           92
6                            Honda Accord           91
7                     Ford Crown Victoria           87
8                  Mercedes-Benz S-Klasse           81
9                           Nissan Altima           76


## Which TV Series Did These Cars Appear The Most?


This code block iterates through the identified top car models to determine the TV series in which each car appeared most frequently, and also lists all series where they made an appearance.

-   **Initialize New Columns**: Two new columns, 'Most Frequent Series' (initialized to `None`) and 'All Appearing Series' (initialized with empty lists), are added to `top_10_cars_df`.
-   **Iterate Through Top Cars**: The code loops through each car model in the `top_10_cars_df`.
-   **Iterate Through TV Shows**: For each car, it then iterates through every TV show in the `tv_show_cars_df`.
-   **Count Car Occurrences in Each Show**: It counts how many times the current `car_model` appears in the 'Cars' list of each `show_title`.
-   **Store Series Appearances**: If a car appears in a show, the `(show_title, count)` is stored in a `series_appearances` list.
-   **Identify Most Frequent Series**: After checking all TV shows for a given car, the `series_appearances` list is sorted by count in descending order. The `show_title` from the top entry is then assigned to the 'Most Frequent Series' column.
-   **List All Appearing Series**: All `show_title`s from the `series_appearances` list are collected and assigned to the 'All Appearing Series' column.
-   **Handle No Appearances**: If a car does not appear in any of the tracked series (though unlikely for top cars), 'N/A' is assigned to 'Most Frequent Series' and an empty list to 'All Appearing Series'.


In [ ]:
from collections import Counter
import pandas as pd # Ensure pandas is imported if not already, for completeness

# Create a copy to avoid SettingWithCopyWarning
top_10_cars_df = car_appearances_df.head(11).copy()

# Add new columns to top_10_cars_df, initialized with empty lists for 'All Appearing Series'
top_10_cars_df['Most Frequent Series'] = None # This is fine as it will hold a single string or 'N/A'
top_10_cars_df['All Appearing Series'] = [[] for _ in range(len(top_10_cars_df))] # Initialize with empty lists

# Iterate through each car in the top_10_cars_df
for idx, car_row in top_10_cars_df.iterrows():
    car_model = car_row['Car Model']
    series_appearances = [] # To store (series_title, count) for the current car

    # Iterate through each TV show in tv_show_cars_df
    for tv_idx, tv_row in tv_show_cars_df.iterrows():
        show_title = tv_row['Title']
        cars_in_show = tv_row['Cars']

        # Count occurrences of the current car_model in the current TV show's car list
        count_in_show = cars_in_show.count(car_model)
        if count_in_show > 0:
            series_appearances.append((show_title, count_in_show))

    if series_appearances:
        # Sort by count in descending order to find the most frequent series
        series_appearances.sort(key=lambda x: x[1], reverse=True)

        # Get the most frequent series
        most_frequent_series = series_appearances[0][0]

        # Get all appearing series (just the titles)
        all_appearing_series_titles = [item[0] for item in series_appearances]

        # Update the top_10_cars_df using .at for single cell assignment
        top_10_cars_df.at[idx, 'Most Frequent Series'] = most_frequent_series
        top_10_cars_df.at[idx, 'All Appearing Series'] = all_appearing_series_titles
    else:
        top_10_cars_df.at[idx, 'Most Frequent Series'] = 'N/A'
        top_10_cars_df.at[idx, 'All Appearing Series'] = []

print("DataFrame 'top_10_cars_df' with series appearance information:")
print(top_10_cars_df)

DataFrame 'top_10_cars_df' with series appearance information:
                                 Car Model  Appearances  \
0                             Toyota Camry          127   
1                              Honda Civic          126   
2                           Toyota Corolla          118   
3                           Ford Econoline          113   
4   Ford Crown Victoria Police Interceptor          102   
5                             Ford Mustang           92   
6                             Honda Accord           91   
7                      Ford Crown Victoria           87   
8                   Mercedes-Benz S-Klasse           81   
9                            Nissan Altima           76   
10                        Chevrolet Impala           75   

                 Most Frequent Series  \
0                                NCIS   
1                          The Rookie   
2                                NCIS   
3                                NCIS   
4                      

Now we can display the results, and download them into `top_10_cars.csv`.

In [ ]:
top_10_cars_df

,Car Model,Appearances,Most Frequent Series,All Appearing Series
0,Toyota Camry,127,NCIS,"[NCIS, The Rookie, Grey's Anatomy, The Lincoln..."
1,Honda Civic,126,The Rookie,"[The Rookie, Supernatural, Veronica Mars, Grey..."
2,Toyota Corolla,118,NCIS,"[NCIS, Homeland, The Rookie, The Wire, Superna..."
3,Ford Econoline,113,NCIS,"[NCIS, Supernatural, The Walking Dead, Law & O..."
4,Ford Crown Victoria Police Interceptor,102,Supernatural,"[Supernatural, NCIS, Fargo, The Walking Dead, ..."
5,Ford Mustang,92,NCIS,"[NCIS, The Rookie, Dexter, Smallville, Strange..."
6,Honda Accord,91,NCIS,"[NCIS, Law & Order: Special Victims Unit, The ..."
7,Ford Crown Victoria,87,NCIS,"[NCIS, Supernatural, Law & Order: Special Vict..."
8,Mercedes-Benz S-Klasse,81,NCIS,"[NCIS, The Rookie, Smallville, Law & Order: Sp..."
9,Nissan Altima,76,Law & Order: Special Victims Unit,"[Law & Order: Special Victims Unit, The Rookie..."


In [ ]:
from google.colab import files

# Convert the DataFrame to a CSV string
csv_data_top_10 = top_10_cars_df.to_csv(index=False)

# Save the CSV data to a file
with open('top_10_cars.csv', 'w') as f:
    f.write(csv_data_top_10)

# Download the file
files.download('top_10_cars.csv')

print("DataFrame 'top_10_cars.csv' downloaded successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DataFrame 'top_10_cars.csv' downloaded successfully.


## Conclusion

This project successfully achieved its core objectives, providing a comprehensive analysis of car model appearances in TV shows using web scraping techniques. Through the systematic approach outlined, the following goals were met:

-   **Environment Setup**: Selenium WebDriver and necessary dependencies were successfully installed and configured for headless execution in Google Colab, demonstrating a robust setup for web scraping.
-   **Demonstrative Scraping**: Example runs for 'The Walking Dead' and 'Breaking Bad' effectively illustrated the core scraping logic, from searching for TV series to extracting car models.
-   **Data Import & Preparation**: A comprehensive list of TV show titles was imported from a Google Sheet, and a DataFrame (`tv_show_cars_df`) was prepared to store the scraped car data.
-   **Automated Data Collection**: An automated process iterated through the TV show list, intelligently navigating IMCDb.org to find and scrape car models for each series. This process incorporated robust error handling, including driver re-initialization and manual link updates for initially failed entries, ensuring maximum data collection.
-   **Car Appearance Analysis**: All collected car models were aggregated, and their total appearance frequencies were accurately counted, resulting in a detailed `car_appearances_df`.
-   **Top Car Identification**: The top 10 most frequently appearing car models (excluding 'unknown' entries) were successfully identified and presented in `top_10_cars_df`.
-   **Series Appearance Breakdown**: For each of the top 10 cars, the most frequent TV series of appearance was determined, along with a complete list of all series featuring that car, enriching the analysis.
-   **Data Export**: The processed data, including the full scraped dataset (`tv_shows_with_cars.csv`), the car appearance counts (`car_appearances.csv`), and the detailed top 10 car analysis (`top_10_cars.csv`), was successfully exported to downloadable CSV files.

Overall, this project provides a valuable dataset and insights into the prevalence of specific car models in popular TV shows, demonstrating the power of automated web scraping and data analysis.